# lightly-train — CB55 few-shot text-line detection (local)

Third detection platform, alongside RF-DETR (`20_detection/rfdetr/`) and
DEIMv2 (`20_detection/deimv2/`): [lightly-train](https://docs.lightly.ai/train/stable/object_detection.html)'s
`train_object_detection`, which fine-tunes a DINOv2/DINOv3-backed LT-DETR
head. Reuses the existing YOLO-format CB55 dataset at
`00_data/DIVA-HisDB/yolo_dataset_CB55/` directly (lightly-train's object
detection task consumes the same `{path, train, val, names}` schema as
Ultralytics) — no new dataset conversion needed.

Run this from the repo's `.venv` (`lightly-train` already installed; see
`20_detection/VERSIONS.md`).

Metrics are computed with our own `torchmetrics.detection.MeanAveragePrecision`
eval (against the exported model's `.predict()`), not TensorBoard:
`train_object_detection`'s TensorBoard writer buffers scalar writes on a
background thread and doesn't flush them until the writer closes (normally
at process exit) — reading the event file back *in the same kernel
process* right after training returns is a real race, verified to
reproducibly miss `val_metric/map` even after 30s of polling (this is what
was crashing the notebook). Rather than depend on that, we just measure
what we care about directly, the same way the held-out test eval already
had to (`train_object_detection` never validates on a held-out test split
anyway).

Pipeline: knobs -> smoke test -> full train -> val + held-out test eval
(own torchmetrics loop) -> results.csv row + viz.


In [1]:
import os
from pathlib import Path
import torch

REPO = Path.cwd()
while not (REPO / '00_data').exists() and REPO.parent != REPO:
    REPO = REPO.parent
assert (REPO / '00_data/DIVA-HisDB/yolo_dataset_CB55').exists(), 'run this notebook from inside the repo'
os.chdir(REPO)

print('cwd:', os.getcwd())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print('vram GB:', round(vram_gb, 1))


cwd: /home/artur/Thesis/fsl-tl-manuscripts
gpu: NVIDIA GeForce RTX 4060 Laptop GPU
vram GB: 8.2


## Knobs

`MODEL` picks the backbone: a ViT path (`dinov2/vits14-ltdetr`) and a
ConvNeXt path (`dinov3/convnext-tiny-ltdetr`) are both available from
`lightly_train.list_models()`-adjacent registries — swap `MODEL` to compare,
same idea as the DEIMv2 backbone matrix. VRAM tiers pick batch size; ViT-S
and ConvNeXt-Tiny both fit comfortably even at 8 GB.


In [2]:
import lightly_train

REPO_ROOT = str(REPO)
HERE = os.path.join(REPO_ROOT, '20_detection/lightly_train')
os.makedirs(HERE, exist_ok=True)

YOLO_DATASET_DIR = os.path.join(REPO_ROOT, '00_data/DIVA-HisDB/yolo_dataset_CB55')
RESULTS_CSV = os.path.join(REPO_ROOT, '20_detection/results.csv')

# MODEL = 'dinov2/vits14-ltdetr'          # ViT path
MODEL = 'dinov3/convnext-tiny-ltdetr'     # CNN path (flip to the ViT one above to compare)

if vram_gb >= 16:
    BATCH_SIZE = 16
elif vram_gb >= 8:
    BATCH_SIZE = 8
else:
    BATCH_SIZE = 4

SMOKE_STEPS = 50
FULL_STEPS = 2000  # ~200 "epochs" at 20 images / batch 4-ish; matches the DEIMv2 few-shot schedule order of magnitude

print('MODEL      :', MODEL)
print('BATCH_SIZE :', BATCH_SIZE)
print('FULL_STEPS :', FULL_STEPS)


MODEL      : dinov3/convnext-tiny-ltdetr
BATCH_SIZE : 8
FULL_STEPS : 2000


## Dataset

`00_data/DIVA-HisDB/yolo_dataset_CB55/dataset.yaml`'s own `path:` field
points at a nonexistent sibling directory (a pre-existing issue in that
file, unrelated to this notebook) — so the `data` dict below is built with
absolute `train`/`val` paths directly instead of loading that YAML,
sidestepping it rather than silently trusting a broken field.


In [3]:
data = {
    'path': YOLO_DATASET_DIR,
    'train': os.path.join(YOLO_DATASET_DIR, 'images/train'),
    'val': os.path.join(YOLO_DATASET_DIR, 'images/val'),
    'names': {0: 'textline'},
}
for split in ('train', 'val'):
    n = len(os.listdir(data[split]))
    print(f'{split}: {n} images at {data[split]}')


train: 20 images at /home/artur/Thesis/fsl-tl-manuscripts/00_data/DIVA-HisDB/yolo_dataset_CB55/images/train
val: 10 images at /home/artur/Thesis/fsl-tl-manuscripts/00_data/DIVA-HisDB/yolo_dataset_CB55/images/val


## Eval helper

Runs the exported model's `.predict()` over an image dir and compares
against the matching YOLO-format labels with
`torchmetrics.detection.MeanAveragePrecision` (same metric class DEIMv2 /
lightly-train use internally, for comparability). Used for both the smoke
sanity check (on val) and the val/test eval after the full run below (no
TensorBoard).


In [4]:
import glob
from PIL import Image
from torchmetrics.detection import MeanAveragePrecision


def eval_map(ckpt_path, images_dir, labels_dir, threshold=0.3):
    model = lightly_train.load_model(ckpt_path)
    model.eval()
    metric = MeanAveragePrecision()
    for img_path in sorted(glob.glob(os.path.join(images_dir, '*.jpg'))):
        img = Image.open(img_path)
        w, h = img.size
        pred = model.predict(img, threshold=threshold)

        lbl_path = os.path.join(labels_dir, os.path.splitext(os.path.basename(img_path))[0] + '.txt')
        gt_boxes, gt_labels = [], []
        for line in open(lbl_path):
            cls, cx, cy, bw, bh = map(float, line.split())
            x0, y0 = (cx - bw / 2) * w, (cy - bh / 2) * h
            x1, y1 = (cx + bw / 2) * w, (cy + bh / 2) * h
            gt_boxes.append([x0, y0, x1, y1])
            gt_labels.append(int(cls))

        metric.update(
            preds=[{
                'boxes': pred['bboxes'].cpu(),
                'scores': pred['scores'].cpu(),
                'labels': pred['labels'].cpu(),
            }],
            target=[{
                'boxes': torch.tensor(gt_boxes, dtype=torch.float32),
                'labels': torch.tensor(gt_labels, dtype=torch.int64),
            }],
        )
    m = metric.compute()
    return float(m['map_50']), float(m['map']), model


## 1. Smoke test

Short real run (`SMOKE_STEPS`), then a sanity-check eval on val: confirms
the eval pipeline produces a real (non-NaN) mAP before committing to the
full run, and prints a full-run time estimate.


In [ ]:
import time

smoke_out = os.path.join(HERE, 'output_smoke')
t0 = time.time()
lightly_train.train_object_detection(
    out=smoke_out,
    data=data,
    model=MODEL,
    steps=SMOKE_STEPS,
    batch_size=BATCH_SIZE,
    overwrite=True,
    logger_args={'tensorboard': None},  # not used -- see notebook intro
)
smoke_elapsed = time.time() - t0
est_full_hours = smoke_elapsed / SMOKE_STEPS * FULL_STEPS / 3600
print(f'smoke: {smoke_elapsed:.1f}s for {SMOKE_STEPS} steps')
print(f'estimated full run ({FULL_STEPS} steps): {est_full_hours:.2f} h')
if est_full_hours > 6:
    print('ESTIMATE EXCEEDS 6h -- lower FULL_STEPS or confirm before running the next cell.')

smoke_ckpt = os.path.join(smoke_out, 'exported_models', 'exported_best.pt')
smoke_map50, smoke_map, _ = eval_map(smoke_ckpt, os.path.join(YOLO_DATASET_DIR, 'images/val'),
                                      os.path.join(YOLO_DATASET_DIR, 'labels/val'))
assert smoke_map50 == smoke_map50, 'smoke val mAP50 is NaN -- eval pipeline broken'  # NaN check
print(f'smoke val mAP50={smoke_map50:.4f}  mAP50-95={smoke_map:.4f} (near-zero this early is expected)')


Using bfloat16 Automatic Mixed Precision (AMP)
Args: {
    "accelerator": "auto",
    "batch_size": 8,
    "checkpoint": null,
    "data": {
        "names": {
            "0": "textline"
        },
        "path": "/home/artur/Thesis/fsl-tl-manuscripts/00_data/DIVA-HisDB/yolo_dataset_CB55",
        "test": null,
        "train": "/home/artur/Thesis/fsl-tl-manuscripts/00_data/DIVA-HisDB/yolo_dataset_CB55/images/train",
        "val": "/home/artur/Thesis/fsl-tl-manuscripts/00_data/DIVA-HisDB/yolo_dataset_CB55/images/val"
    },
    "devices": "auto",
    "float32_matmul_precision": "auto",
    "loader_args": null,
    "logger_args": {
        "tensorboard": null
    },
    "model": "dinov3/convnext-tiny-ltdetr",
    "model_args": null,
    "num_nodes": 1,
    "num_workers": "auto",
    "out": "/home/artur/Thesis/fsl-tl-manuscripts/20_detection/lightly_train/output_smoke",
    "overwrite": true,
    "precision": "bf16-mixed",
    "resume_interrupted": false,
    "reuse_class_head": false

## 2. Full train

Trains for `FULL_STEPS`, exports the best-by-mAP checkpoint to
`out/exported_models/exported_best.pt`.


In [ ]:
full_out = os.path.join(HERE, 'output_full')
t0 = time.time()
lightly_train.train_object_detection(
    out=full_out,
    data=data,
    model=MODEL,
    steps=FULL_STEPS,
    batch_size=BATCH_SIZE,
    overwrite=True,
    logger_args={'tensorboard': None},
)
full_elapsed = time.time() - t0
print(f'full train: {full_elapsed/60:.1f} min')


## 3. Val + held-out test eval

`train_object_detection` validates internally during training (that's how
the best checkpoint is picked) but doesn't hand the number back to Python,
so this re-evaluates val with the exported best checkpoint (should roughly
match what training reported), plus the held-out `images/test` split it
never saw. `epochs_to_best` comes straight from the checkpoint's own `step`
field.


In [ ]:
best_ckpt_file = os.path.join(full_out, 'checkpoints', 'best.ckpt')
best_exported = os.path.join(full_out, 'exported_models', 'exported_best.pt')
best_step = torch.load(best_ckpt_file, map_location='cpu', weights_only=False)['step']

val_map50, val_map, _ = eval_map(best_exported, os.path.join(YOLO_DATASET_DIR, 'images/val'),
                                  os.path.join(YOLO_DATASET_DIR, 'labels/val'))

TEST_IMG_DIR = os.path.join(YOLO_DATASET_DIR, 'images/test')
TEST_LBL_DIR = os.path.join(YOLO_DATASET_DIR, 'labels/test')
test_map50, test_map, model = eval_map(best_exported, TEST_IMG_DIR, TEST_LBL_DIR)

print(f'best_step={best_step}  val mAP50={val_map50:.4f}/{val_map:.4f}  test mAP50={test_map50:.4f}/{test_map:.4f}')


## 4. Append results.csv row + save 3 viz predictions

In [ ]:
import csv

total_params = sum(p.numel() for p in model())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

row = {
    'platform': 'lightly_train',
    'model': 'ltdetr',
    'backbone': MODEL,
    'pretrain': f'{MODEL} stock pretrain',
    'resolution': 'model default',
    'epochs_to_best': best_step,
    'val_mAP50': val_map50,
    'val_mAP50_95': val_map,
    'test_mAP50': test_map50,
    'test_mAP50_95': test_map,
    'params_total': total_params,
    'params_trainable': trainable_params,
    'notes': f'lightly-train train_object_detection, steps={FULL_STEPS}, batch={BATCH_SIZE}',
}
header = ['platform', 'model', 'backbone', 'pretrain', 'resolution', 'epochs_to_best',
          'val_mAP50', 'val_mAP50_95', 'test_mAP50', 'test_mAP50_95',
          'params_total', 'params_trainable', 'notes']
write_header = not os.path.exists(RESULTS_CSV)
with open(RESULTS_CSV, 'a', newline='') as f:
    w = csv.DictWriter(f, fieldnames=header)
    if write_header:
        w.writeheader()
    w.writerow(row)
print('appended:', row)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

viz_dir = os.path.join(HERE, 'viz')
os.makedirs(viz_dir, exist_ok=True)
test_imgs = sorted(glob.glob(os.path.join(TEST_IMG_DIR, '*.jpg')))[:3]

fig, axes = plt.subplots(1, len(test_imgs), figsize=(18, 6))
for ax, img_path in zip(axes, test_imgs):
    img = Image.open(img_path)
    pred = model.predict(img, threshold=0.5)

    # per-image save (own figure, not the combined display grid below)
    single_fig, single_ax = plt.subplots(figsize=(8, 10))
    single_ax.imshow(img)
    for box in pred['bboxes'].cpu().numpy():
        x0, y0, x1, y1 = box
        single_ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='lime', linewidth=1.5))
    single_ax.axis('off')
    single_fig.savefig(os.path.join(viz_dir, os.path.basename(img_path)), bbox_inches='tight')
    plt.close(single_fig)

    # combined display grid
    ax.imshow(img)
    for box in pred['bboxes'].cpu().numpy():
        x0, y0, x1, y1 = box
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='lime', linewidth=1.5))
    ax.set_title(os.path.basename(img_path))
    ax.axis('off')
plt.tight_layout()
plt.show()
print('viz saved to', viz_dir)
